# A large scale analysis of conversation data across myriad spoken corpora

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [ ]:
DATA_LOC = 'data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

In [ ]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [ ]:
fs = grab_all_files(DATA_PATH, '.tsv')
fs

Iteratively import dataframes and editing them

In [ ]:
df = []

In [ ]:
for f in tqdm(fs):
    df += [pd.read_table(f, sep='\t')]
    
    if ('/cha-' in f) or ('grace-' in f):
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]
   
    elif '/candor-' in f:
        df[-1]['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df[-1]['conversation_id'])]
    
    else:
        if 'conversation_id' in df[-1].columns:
            0
        else:
            df[-1]['conversation_id'] = f
        
    df[-1] = df[-1].loc[
        (df[-1]['nx'] >= 5) 
        & (df[-1]['ny'] >= 5) 
        & (df[-1]['x_turn_id'] >= 5)
    ]
    

In [ ]:
df = pd.concat(df, ignore_index=True)
df.shape

In [ ]:
df['conversation_id'].nunique()

In [ ]:
# df['avgHxy'] = df['Hxy'] / df['nx']

In [ ]:
df['dyad'] = df['x_speaker'] + '-' + df['y_speaker']

In [ ]:
df['turn_delta'] = (df['y_turn_id'] - df['x_turn_id'])
# df['turn_delta'] = (df['y_start'] - df['x_start'])

In [ ]:
df['x_speaker_turn'] = df['x_speaker'] + '-' + df['x_turn_id'].astype(str)

In [ ]:
df.head()

## Copy and concatenate reversed data!

In [ ]:
# df2 = df.copy()

In [ ]:
# new_column_names = dict()
# for k in list(df2):
#     if k.startswith('x_'):
#         new_column_names[k] = k.replace('x_', 'y_')
#     elif k.startswith('y_'):
#         new_column_names[k] = k.replace('y_', 'x_')
#     elif k == 'Hxy':
#         new_column_names[k] = 'Hyx'
#     elif k == 'Hyx':
#         new_column_names[k] = 'Hxy'
#     elif k == 'nx':
#         new_column_names[k] = 'ny'
#     elif k == 'ny':
#         new_column_names[k] = 'nx'
#     else:
#         new_column_names[k]=k

In [ ]:
# df2 = df2.rename(columns=new_column_names)
# df2.head()

In [ ]:
# df2['x_speaker_turn'] = df2['x_speaker'] + '-' + df2['x_turn_id'].astype(str)

In [ ]:
# df = pd.concat([df, df2], ignore_index=True)

In [ ]:
# df.shape

## Convert values to numeric tags

In [ ]:
convert_to_numeric_id = [
    'x_speaker', 'y_speaker', 'dyad', 'x_speaker_turn'
]

In [ ]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}
    
    if isinstance(col,list):
        for c  in col:
            df[c] = [conversion[v] for v in tqdm(df[c])]
    
    else:
        df[col] = [conversion[v] for v in tqdm(df[col])]
    

## LME Regression: Predicting CE

In [ ]:
from tqdm import tqdm
import statsmodels.formula.api as smf

In [ ]:
df_ = df

In [ ]:
##########################################
## Main model
##########################################
model = "Hxy ~ nx + ny + turn_delta*self"
##########################################

In [ ]:
start = dt.now()
# md = smf.mixedlm(model, data=df_, groups=df_['x_speaker_turn'])
md = smf.mixedlm(model, data=df_, groups=df_['dyad'])
mdf = md.fit()
print('completed in:', dt.now()-start)

In [ ]:
# mdf.summary()

In [ ]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

In [ ]:
reporting.to_csv(os.path.join(DATA_LOC, 'lme-results.csv'), encoding='utf-8')

In [ ]:
reporting['Var'] = reporting.index.values

with open(os.path.join(DATA_LOC, 'lme-results.txt'), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()

## Create a resid

In [ ]:
##########################################
## A set of resids to show ∇H / t
##########################################

model = "Hxy ~ nx + ny"
##########################################

In [ ]:
start = dt.now()
# md = smf.mixedlm(model, data=df_, groups=df_['x_speaker_turn'])
md = smf.mixedlm(model, data=df_, groups=df_['dyad'])
mdf = md.fit()
print('completed in:', dt.now()-start)

In [ ]:
# mdf.summary()

In [ ]:
df_['resid'] = mdf.resid

## Vis & Other Tests

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# sns.lineplot(
#     data=df_.loc[
#         ~df_['self']
#         & (df_['turn_delta'] > 0)
#         & (df_['turn_delta'] < 40)
#     ], 
#     y='resid', 
#     x='turn_delta', 
# )
# plt.show()

In [ ]:
# sns.lineplot(
#     data=df_.loc[
#         df_['self'] 
#         & (df_['turn_delta'] > 0)
#         & (df_['turn_delta'] < 40)
#     ], 
#     y='resid', 
#     x='turn_delta', 
#     color='orange'
# )
# plt.show()

In [ ]:
ax = sns.lineplot(
    data=df_.loc[
        (df_['turn_delta'] > 0)
        & (df_['turn_delta'] < 40)
    ], 
    y='resid', 
    x='turn_delta', 
    hue='self',
    legend=False
)
plt.show()

### Creating an interactive plot using px to start, and then converting to GO :)

In [ ]:
import plotly.tools as tls
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
fig = tls.mpl_to_plotly(ax.figure)

In [ ]:
names = ['other', 'self']
main_line_colors = ['#1f77b4','#ff7f0e']
ribbon_colors = ['rgba(31, 119, 180, 0.2)', 'rgba(255, 127, 14, 0.2)']

In [ ]:
new_fig = go.Figure()

In [ ]:
for i,trace in enumerate(fig.data):
    trace['name'] = names[i]
    # print(trace)
    trace['showlegend'] = True
    trace['line']['color'] = main_line_colors[i]
    trace['legendgroup'] = names[i]
    # trace['legendgrouptitle'] = names[i]
    new_fig.add_trace(trace)
    
    # upper bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=trace['x'],
            y=(np.array(trace['y']) + (mdf.bse['Group Var']*2)).tolist(),
            mode='lines',
            line=dict(color=main_line_colors[i],width=0),
            name=names[i]+'_upper_bound',
            showlegend=False,
            legendgroup=names[i]
        )
    )
    
    
    # lower bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=trace['x'],
            y=(np.array(trace['y']) - (mdf.bse['Group Var']*2)).tolist(),
            mode='lines',
            line=dict(color=main_line_colors[i],width=0),
            name=names[i]+'_lower_bound',
            fill='tonexty',
            fillcolor=ribbon_colors[i],
            showlegend=False,
            legendgroup=names[i]
        )
    )
    
# new_fig.update_layout(showlegend=True)
new_fig.update_layout(template='plotly')
new_fig.show()  

In [ ]:
new_fig.write_html(os.path.join(VIS_PATH, 'entropy-delta.html'))

### Individual conversations in a subplot

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
def create_subplot(
        dataframe, 
        new_fig, 
        row, 
        col, 
        names=names, 
        main_line_colors=main_line_colors, 
        ribbon_colors=ribbon_colors
):
    
    ax = sns.lineplot(
        data=dataframe, 
        y='resid', 
        x='turn_delta', 
        hue='self',
        legend=False
    )
    
    fig = tls.mpl_to_plotly(ax.figure)
    
    for i,trace in enumerate(fig.data):
        trace['name'] = names[i]
        # print(trace)
        trace['showlegend'] = False
        if (row==1) and (col==1):
            trace['showlegend'] = True
        trace['line']['color'] = main_line_colors[i]
        trace['legendgroup'] = names[i]
        # trace['legendgrouptitle'] = names[i]
        new_fig.add_trace(trace, row=row, col=col)
        
        # upper bounds for ribbon
        new_fig.add_trace(
            go.Scatter(
                x=trace['x'],
                y=(np.array(trace['y']) + mdf.bse['Group Var']).tolist(),
                mode='lines',
                line=dict(color=main_line_colors[i],width=0),
                name=names[i]+'_upper_bound',
                showlegend=False,
                legendgroup=names[i]
            ), row=row, col=col
        )
        
        # lower bounds for ribbon
        new_fig.add_trace(
            go.Scatter(
                x=trace['x'],
                y=(np.array(trace['y']) - mdf.bse['Group Var']).tolist(),
                mode='lines',
                line=dict(color=main_line_colors[i],width=0),
                name=names[i]+'_lower_bound',
                fill='tonexty',
                fillcolor=ribbon_colors[i],
                showlegend=False,
                legendgroup=names[i]
            ), row=row, col=col
        )

#### Actually creating the figure

In [ ]:
conversation_id_col = 'conversation_id'

In [ ]:
k_conversations = 12
k_columns = 3
k_rows = int(np.ceil(k_conversations/k_columns))

In [ ]:
assignments = sum([[(row+1,col+1) for col in range(k_columns)] for row in range(k_rows)], [])

In [ ]:
candorsel = df_['conversation_id'].apply(lambda x: 'candor-' in x)
phonesel = df_['conversation_id'].apply(lambda x: x.startswith('call' ))
# teachersel = df_['conversation_id'].apply(lambda x: ('instruction-' in x.lower()))
teachersel = df_['conversation_id'].apply(lambda x: ('graesser-' in x.lower()) or ('med_school-' in x.lower()))
bncsel = df_['conversation_id'].apply(lambda x: 'cabnc-' in x.lower())

In [ ]:
# Select a number of conversations (k_converastions total)
# subdata = np.random.choice(df_[conversation_id_col].unique(), size=(k_conversations,), replace=False)

# Zoom corpus
subdata = np.random.choice(df_[conversation_id_col].loc[candorsel].unique(), size=(k_columns,), replace=False).tolist()

# Telephone corpus
subdata += np.random.choice(df_[conversation_id_col].loc[phonesel].unique(), size=(k_columns,), replace=False).tolist()

# Teaching/Tutoring Corpus
subdata += np.random.choice(df_[conversation_id_col].loc[teachersel].unique(), size=(k_columns,), replace=False).tolist()

# BNC/Naturalistic corpus
subdata += np.random.choice(df_[conversation_id_col].loc[bncsel].unique(), size=(k_columns,), replace=False).tolist()

In [ ]:
tracking_doc = os.path.join(VIS_PATH,'tracking.txt')

if (not os.path.exists(tracking_doc)):
    f = open(tracking_doc, 'w')
    f.write('{} | {}\n'.format(dt.now(), str(subdata)))
    f.close()

else:
    f = open(tracking_doc, 'a')
    f.write('{} | {}\n'.format(dt.now(), str(subdata)))
    f.close()

In [ ]:
subplot_fig = make_subplots(
    rows=k_rows,cols=k_columns, 
    row_titles=['Zoom', 'Telephone', 'Instruction', 'BNC/Naturalistic']
)

In [ ]:
for (i, conversation) in tqdm(enumerate(subdata)):
    df__ = df_.loc[
        df[conversation_id_col].isin([conversation])
        & (df_['turn_delta'] > 0)
        & (df_['turn_delta'] < 40)
    ]
    
    create_subplot(
        dataframe=df__,
        new_fig=subplot_fig,
        row=assignments[i][0],
        col=assignments[i][1]
    )   

In [ ]:
subplot_fig.update_layout(template='plotly')
subplot_fig.show()

In [ ]:
subplot_fig.write_html(os.path.join(VIS_PATH, 'entropy-delta-individual-conversations.html'))